# STRATA DNS Field Viewer — Colab

**R4P50 · Hsst simulation · t = 117.315250**

Pull a **subdomain** of the R4P50 Zarr store from OLCF Constellation over
**Globus HTTPS**, reconstruct it locally, save it as a `.npy`, and visualize it.

Run the steps below in order:

1. **Authenticate with Globus** — installs dependencies, loads the libraries, and logs you in.
2. **Download data** — pick a resolution level, variable and x/y/z index ranges; only the shard files your selection touches are transferred, and `zarr` reassembles the requested subdomain into `./strata_data/`.
3. **Visualize** — point at any saved `.npy` and slice through it.

**Before you begin:** a free [Globus account](https://www.globus.org/) is
required (sign in with Google or your institution). If the interactive widgets
don't render, run once:
`from google.colab import output; output.enable_custom_widget_manager()`

---
## Step 1 — Authenticate with Globus

Run the cell below to install dependencies, load the libraries, and log in.
Open the URL it prints, log in with your Globus account, and paste the
authorisation code back here. The login happens on Globus's own pages, so
your OLCF/institutional credentials never touch this notebook.

In [ ]:
#@title Step 1: Authenticate with Globus
!pip install -q "zarr>=3" globus-sdk requests ipywidgets

import base64, glob, itertools, math, os, re

import numpy as np
import requests
import zarr
import globus_sdk
import ipywidgets as widgets
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import display, HTML

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

# --- dataset / store configuration -------------------------------------------
# R4P50 sharded Zarr v3 store. Inner chunk is 64^3; sharded levels group inner
# chunks into larger shard files. On coarse (unsharded) levels the file
# granularity is the inner chunk itself.
LEVEL_INFO = [
    dict(label="0 - full res  16000x8000x4000",  shape=(16000, 8000, 4000), shard=(1024, 512, 256), chunk=(64, 64, 64), sub="",   available=True),
    dict(label="1 - 2x  8000x4000x2000",         shape=(8000,  4000, 2000), shard=(512,  256, 128), chunk=(64, 64, 64), sub="/1", available=False),
    dict(label="2 - 4x  4000x2000x1000",         shape=(4000,  2000, 1000), shard=(256,  128,  64), chunk=(64, 64, 64), sub="/2", available=False),
    dict(label="3 - 8x  2000x1000x500",          shape=(2000,  1000,  500), shard=None,             chunk=(64, 64, 64), sub="/3", available=False),
    dict(label="4 - 16x  1000x500x250",          shape=(1000,   500,  250), shard=None,             chunk=(64, 64, 64), sub="/4", available=False),
    dict(label="5 - 32x  500x250x125",           shape=(500,    250,  125), shard=None,             chunk=(64, 64, 64), sub="/5", available=False),
]
VARIABLE_INFO = {
    "r": "r  -  density (buoyancy)",
    "u": "u  -  streamwise velocity",
    "v": "v  -  spanwise velocity",
    "w": "w  -  vertical velocity",
}
AVAILABLE_VARS = ["r"]          # add u, v, w as those conversions finish
CMAP = "RdBu_r"

# Both persist for the life of the Colab runtime.
CACHE_ROOT     = "./strata_cache"     # fetched shards land here
SAVE_DIR       = "./strata_data"      # reconstructed .npy cubes land here
DATAURI_MAX_MB = 50                   # inline browser-download link up to this
MAX_FETCH_GB   = 6                    # RAM guard for free Colab (~12 GB)

# --- Globus HTTPS settings (per-user OAuth + HTTPS GET) --------------------
CLIENT_ID       = "d47db6dc-0428-4076-9a6e-31927d7c7704"                # registered Globus Native app id
COLLECTION_ID   = "36d521b3-c182-4071-b7d5-91db5d380d42"                # mapped collection UUID
HTTPS_BASE      = "https://m-36da98.abb646.36fe.data.globus.org"       # HTTPS-mapped collection endpoint
COLLECTION_PATH = "/lustre/orion/cfd135/proj-shared/zarrTest/zarr/"     # store root on the collection

os.makedirs(CACHE_ROOT, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)
print("shard cache :", os.path.abspath(CACHE_ROOT))
print("saved cubes :", os.path.abspath(SAVE_DIR))

# --- targeted shard transfer over Globus HTTPS (inlined; no repo needed) ----
def shard_rel_paths(var, grid, sel):
    """POSIX rel paths of the metadata + data files a selection needs.

    `grid` is the shard shape (sharded level) or inner-chunk shape (coarse
    level); on-disk files live at <var>/c/<i>/<j>/<k>. Enumerating indices
    (rather than listing a directory) is what lets HTTPS GET work.
    `sel` is (x0, x1, y0, y1, z0, z1) in level-local indices.
    """
    paths = ["zarr.json", var + "/zarr.json"]
    x0, x1, y0, y1, z0, z1 = sel
    gx, gy, gz = grid
    for i in range(x0 // gx, math.ceil(x1 / gx)):
        for j in range(y0 // gy, math.ceil(y1 / gy)):
            for k in range(z0 // gz, math.ceil(z1 / gz)):
                paths.append(f"{var}/c/{i}/{j}/{k}")
    return paths


class GlobusHttpsBackend:
    """HTTPS GET of individual shard files, authorised by a Globus token."""

    def __init__(self, base_url, token):
        self.base_url = base_url.rstrip("/")
        self.token = token
        self._sess = requests.Session()

    def fetch(self, rel_paths, local_root):
        headers = {"Authorization": f"Bearer {self.token}"} if self.token else {}
        for rel in rel_paths:
            url = f"{self.base_url}/{rel}?download=1"
            dst = os.path.join(local_root, rel)
            r = self._sess.get(url, headers=headers, stream=True)
            if r.status_code == 404:
                continue                      # shard absent (all-fill) -> skip
            r.raise_for_status()
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with open(dst, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)


def ensure_local(cache_root, backend, var, grid, sel):
    """Fetch the files a selection needs into cache_root (skip present shards;
    always refresh tiny metadata). Returns cache_root to open with zarr."""
    rel = shard_rel_paths(var, grid, sel)
    missing = [r for r in rel if not os.path.exists(os.path.join(cache_root, r))]
    meta = [r for r in rel if r.endswith("zarr.json")]
    need = sorted(set(missing) | set(meta))
    if need:
        backend.fetch(need, cache_root)
    return cache_root


def fetch_metadata(cache_root, backend, var):
    """Fetch only the two zarr.json files -- a lightweight auth/path check."""
    backend.fetch(["zarr.json", var + "/zarr.json"], cache_root)
    return cache_root


# --- shared plotting helpers (used by both the download and visualize widgets)
W = widgets.Layout

_BOX_EDGES = [(0, 4), (1, 5), (2, 6), (3, 7),
              (0, 2), (1, 3), (4, 6), (5, 7),
              (0, 1), (2, 3), (4, 5), (6, 7)]


def _draw_box(ax, x0, x1, y0, y1, z0, z1, **kw):
    pts = np.array(list(itertools.product([x0, x1], [y0, y1], [z0, z1])))
    for a, b in _BOX_EDGES:
        ax.plot(*zip(pts[a], pts[b]), **kw)


# --- authenticate ------------------------------------------------------------
HTTPS_SCOPE = f"https://auth.globus.org/scopes/{COLLECTION_ID}/https"

# Force authentication with your OLCF identity (PIN + RSA SecurID) instead of
# silently reusing an existing Globus session (e.g. a Google login). Without
# this you just get a token for whatever identity is already logged in -> the
# OLCF mapped collection then rejects it with 403.
#   * prompt="login"                  -> do not reuse the current session
#   * session_required_single_domain  -> require an identity in the OLCF SSO
#                                        domain (this triggers the RSA prompt)
OLCF_DOMAIN = "sso.ccs.ornl.gov"                 # OLCF SSO domain (all OLCF users)

_client = globus_sdk.NativeAppAuthClient(CLIENT_ID)
_client.oauth2_start_flow(requested_scopes=HTTPS_SCOPE)

_qp = {"prompt": "login", "session_required_single_domain": OLCF_DOMAIN}
_authorize_url = _client.oauth2_get_authorize_url(query_params=_qp)

print("Open this URL and log in WITH YOUR OLCF IDENTITY (PIN + RSA token),")
print("then paste the code below:\n")
print(_authorize_url)

_code = input("\nAuthorisation code: ").strip()
_tok = _client.oauth2_exchange_code_for_tokens(_code)

HTTPS_TOKEN = None
for _rs, _data in _tok.by_resource_server.items():
    if _rs != "auth.globus.org":
        HTTPS_TOKEN = _data["access_token"]
        break

assert HTTPS_TOKEN, "Authentication failed: no HTTPS token received."
print("Authentication successful.")

---
## Step 2 — Download data

Pick a resolution level, variable and x/y/z index ranges. Only the shard
files your selection touches are transferred; `zarr` reassembles the
requested subdomain, which is saved to `./strata_data/`.

In [ ]:
#@title Step 2: Download data
# --- backend factory, transfer + reconstruct, estimate, wireframe ----------
def _grid(info):
    """File-granularity shape: shard shape if sharded, else inner-chunk shape."""
    return info["shard"] or info["chunk"]


def make_backend(info):
    base = HTTPS_BASE.rstrip("/") + COLLECTION_PATH + info["sub"]
    return GlobusHttpsBackend(base, HTTPS_TOKEN)


def resolve_store(level_idx, var, sel):
    """Fetch just the shards this selection touches; return a store path to open."""
    info = LEVEL_INFO[level_idx]
    cache = CACHE_ROOT + info["sub"]
    ensure_local(cache, make_backend(info), var, _grid(info), sel)
    return cache


def check_connection(level_idx, var):
    """Validate auth + path by fetching metadata only (no shards)."""
    import time
    try:
        info = LEVEL_INFO[level_idx]
        cache = CACHE_ROOT + info["sub"]
        t0 = time.perf_counter()
        fetch_metadata(cache, make_backend(info), var)
        zarr.open_group(cache, mode="r")[var]
        return True, f"OK - fetched metadata over HTTPS in {time.perf_counter() - t0:.1f}s"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


def transfer_estimate(x0, x1, y0, y1, z0, z1, level_idx):
    """Return (n_shards_or_None, xfer_mb, data_mb)."""
    info = LEVEL_INFO[level_idx]
    data_mb = (x1 - x0) * (y1 - y0) * (z1 - z0) * 4 / 1e6
    if info["shard"] is None:
        return None, data_mb, data_mb
    sx, sy, sz = info["shard"]
    n = ((math.ceil(x1 / sx) - math.floor(x0 / sx)) *
         (math.ceil(y1 / sy) - math.floor(y0 / sy)) *
         (math.ceil(z1 / sz) - math.floor(z0 / sz)))
    return n, n * sx * sy * sz * 4 / 1e6, data_mb


def format_estimate(x0, x1, y0, y1, z0, z1, level_idx):
    n, xfer_mb, data_mb = transfer_estimate(x0, x1, y0, y1, z0, z1, level_idx)
    lines = []
    if n is not None:
        lines.append(f"<b>Transfer:</b> {n} shard{'s' if n != 1 else ''}  "
                     f"(~{xfer_mb / 1024:.2f} GB uncompressed)")
    lines.append(f"<b>Decompressed data:</b> {data_mb:.0f} MB")
    return "<br>".join(lines)


def draw_wireframe(level_idx, sel):
    nx, ny, nz = LEVEL_INFO[level_idx]["shape"]
    x0, x1, y0, y1, z0, z1 = sel
    fig = plt.figure(figsize=(5.2, 4.2))
    ax = fig.add_subplot(111, projection="3d")
    _draw_box(ax, 0, nx, 0, ny, 0, nz, color="0.65", lw=0.8)
    _draw_box(ax, x0, x1, y0, y1, z0, z1, color="tab:red", lw=2.5)
    ax.set_xlim(0, nx); ax.set_ylim(0, ny); ax.set_zlim(0, nz)
    ax.set_box_aspect((nx, ny, nz))
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_title("Selection within full domain", fontsize=10)
    ax.view_init(elev=22, azim=-58)
    return fig


def save_cube(data, sel, level_idx, var):
    x0, x1, y0, y1, z0, z1 = sel
    fname = f"{var}_L{level_idx}_x{x0}-{x1}_y{y0}-{y1}_z{z0}-{z1}.npy"
    path = os.path.join(SAVE_DIR, fname)
    np.save(path, data)
    return path, fname, os.path.getsize(path) / 1e6


def download_link_html(path, fname, size_mb):
    parts = [f"Saved to <code>{path}</code> &nbsp;({size_mb:.1f} MB)"]
    if size_mb <= DATAURI_MAX_MB:
        with open(path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        parts.append(f"<a download='{fname}' "
                     f"href='data:application/octet-stream;base64,{b64}' "
                     f"style='font-weight:bold'>Download {fname} to your computer</a>")
    else:
        parts.append(f"<i>Exceeds the {DATAURI_MAX_MB} MB in-browser limit; use "
                     f"the Files panel or <code>google.colab.files.download('{path}')</code>.</i>")
    return "<br>".join(parts)


# --- download widget -------------------------------------------------------
w_level = widgets.Dropdown(
    options=[(i["label"], k) for k, i in enumerate(LEVEL_INFO) if i["available"]],
    description="Level:", layout=W(width="480px"), style={"description_width": "60px"})
w_var = widgets.Dropdown(
    options=[(VARIABLE_INFO[v], v) for v in AVAILABLE_VARS],
    description="Variable:", layout=W(width="280px"), style={"description_width": "75px"})


def _range_slider(label, maxval, default_hi):
    return widgets.IntRangeSlider(
        value=[0, min(default_hi, maxval)], min=0, max=maxval, step=64,
        description=label, continuous_update=False,
        layout=W(width="620px"), style={"description_width": "25px"})


nx0, ny0, nz0 = LEVEL_INFO[0]["shape"]
w_x = _range_slider("x", nx0, 1024)
w_y = _range_slider("y", ny0,  512)
w_z = _range_slider("z", nz0,  256)
w_full = widgets.Checkbox(value=False, description="Download entire box (full range)",
                          indent=False, layout=W(width="340px"))
w_estimate = widgets.HTML()
w_wire     = widgets.Output()
w_fetch    = widgets.Button(description="Fetch & Download", button_style="primary",
                            icon="download", layout=W(width="180px"))
w_check    = widgets.Button(description="Test connection", icon="link", layout=W(width="160px"))
w_status   = widgets.Output()


def _ranges(level_idx):
    """Read sliders; a zero-width range [i, i] becomes a single plane [i, i+1]."""
    nx, ny, nz = LEVEL_INFO[level_idx]["shape"]
    out = []
    for w, n in ((w_x, nx), (w_y, ny), (w_z, nz)):
        a, b = w.value
        if b <= a:
            a = min(a, n - 1); b = a + 1
        out += [a, b]
    return out


def _update_preview(*_):
    sel = _ranges(w_level.value)
    w_estimate.value = format_estimate(*sel, w_level.value)
    with w_wire:
        w_wire.clear_output(wait=True)
        fig = draw_wireframe(w_level.value, sel); display(fig); plt.close(fig)


def on_level_change(change):
    nx, ny, nz = LEVEL_INFO[change["new"]]["shape"]
    for w, n, d in [(w_x, nx, 1024), (w_y, ny, 512), (w_z, nz, 256)]:
        w.max = n
        w.value = [0, n] if w_full.value else [0, min(d, n)]
    _update_preview()


def on_full_change(change):
    full = change["new"]
    nx, ny, nz = LEVEL_INFO[w_level.value]["shape"]
    for w, n in ((w_x, nx), (w_y, ny), (w_z, nz)):
        w.disabled = full
        if full:
            w.value = [0, n]
    _update_preview()


def on_check(_):
    with w_status:
        w_status.clear_output(wait=True); print("Testing HTTPS connection...")
    ok, msg = check_connection(w_level.value, w_var.value)
    colour = "#0a0" if ok else "#b00"
    with w_status:
        w_status.clear_output(wait=True)
        display(HTML(f"<span style='color:{colour}'>{msg}</span>"))


def on_fetch(_):
    sel = _ranges(w_level.value)
    x0, x1, y0, y1, z0, z1 = sel
    lvl, var = w_level.value, w_var.value

    n_shards, _, data_mb = transfer_estimate(*sel, lvl)
    if data_mb / 1024 > MAX_FETCH_GB:
        with w_status:
            w_status.clear_output(wait=True)
            display(HTML(f"<span style='color:#b00'><b>Selection too large:</b> "
                         f"{data_mb / 1024:.1f} GB exceeds the {MAX_FETCH_GB} GB limit.<br>"
                         f"Narrow the ranges or pick a coarser level.</span>"))
        return

    with w_status:
        w_status.clear_output(wait=True)
        n_txt = f"{n_shards} shard{'s' if n_shards != 1 else ''}" if n_shards else "data"
        print(f"Fetching {n_txt} over Globus HTTPS...")

    store = resolve_store(lvl, var, sel)              # targeted shard transfer
    root  = zarr.open_group(store, mode="r")
    data  = np.asarray(root[var][x0:x1, y0:y1, z0:z1])  # zarr reconstructs subdomain
    path, fname, size_mb = save_cube(data, sel, lvl, var)

    with w_status:
        w_status.clear_output(wait=True)
        print(f"Reconstructed  shape={data.shape}   "
              f"min={data.min():.4g}   max={data.max():.4g}")
        display(HTML(download_link_html(path, fname, size_mb)))


for w in (w_x, w_y, w_z):
    w.observe(_update_preview, names="value")
w_level.observe(on_level_change, names="value")
w_full.observe(on_full_change, names="value")
w_fetch.on_click(on_fetch)
w_check.on_click(on_check)

panel = widgets.VBox([
    widgets.HTML("<b>Choose resolution, variable and index ranges</b>"),
    widgets.HBox([w_level, w_var]),
    w_x, w_y, w_z, w_full,
    widgets.HTML("<hr style='margin:6px 0'>"),
    widgets.HBox([
        widgets.VBox([w_estimate, widgets.HTML("<br>"),
                      widgets.HBox([w_fetch, w_check]), w_status],
                     layout=W(width="420px")),
        w_wire,
    ]),
], layout=W(padding="12px", max_width="1100px"))

_update_preview()
display(panel)

---
## Step 3 — Visualize

Reads whatever `.npy` cubes live in `./strata_data/`. Pick one, load it, and
slice through it. (Press **Refresh list** after fetching a new cube above.)

In [ ]:
#@title Step 3: Visualize
# --- single-slice viewer helpers -------------------------------------------
# orientation -> (perpendicular axis letter, index into data.shape)
PERP_AXIS = {"xy": ("z", 2), "yz": ("x", 0), "xz": ("y", 1)}


def parse_offsets(fname):
    """Recover (var, x0, y0, z0) from a saved filename; fall back to zeros."""
    base = os.path.basename(fname)
    m = re.match(r"(?P<var>[A-Za-z0-9]+)_L\d+_x(?P<x>\d+)-\d+_y(?P<y>\d+)-\d+_z(?P<z>\d+)-\d+", base)
    if m:
        return m["var"], int(m["x"]), int(m["y"]), int(m["z"])
    return base, 0, 0, 0


def plot_single_slice(data, x0, y0, z0, var, orient, pos, vlim):
    lo, hi = vlim
    kw = dict(cmap=CMAP, vmin=lo, vmax=hi, origin="lower", aspect="auto")
    label = VARIABLE_INFO.get(var, var)
    fig, ax = plt.subplots(figsize=(8, 5))
    if orient == "xy":
        im = ax.imshow(data[:, :, pos].T, **kw)
        ax.set(xlabel=f"x  (off {x0})", ylabel=f"y  (off {y0})",
               title=f"{label}  -  xy plane at z = {z0 + pos}")
    elif orient == "yz":
        im = ax.imshow(data[pos, :, :].T, **kw)
        ax.set(xlabel=f"y  (off {y0})", ylabel=f"z  (off {z0})",
               title=f"{label}  -  yz plane at x = {x0 + pos}")
    else:
        im = ax.imshow(data[:, pos, :].T, **kw)
        ax.set(xlabel=f"x  (off {x0})", ylabel=f"z  (off {z0})",
               title=f"{label}  -  xz plane at y = {y0 + pos}")
    plt.colorbar(im, ax=ax, label=label); plt.tight_layout(); plt.show()


def draw_slice_wireframe(shape, orient, pos):
    nx, ny, nz = shape
    if orient == "xy":
        z = pos; corners = [(0, 0, z), (nx, 0, z), (nx, ny, z), (0, ny, z)]
    elif orient == "yz":
        x = pos; corners = [(x, 0, 0), (x, ny, 0), (x, ny, nz), (x, 0, nz)]
    else:
        y = pos; corners = [(0, y, 0), (nx, y, 0), (nx, y, nz), (0, y, nz)]
    fig = plt.figure(figsize=(4.6, 4.0))
    ax = fig.add_subplot(111, projection="3d")
    _draw_box(ax, 0, nx, 0, ny, 0, nz, color="0.6", lw=0.9)
    ax.add_collection3d(Poly3DCollection([corners], alpha=0.35,
                        facecolor="tab:red", edgecolor="tab:red"))
    ax.set_xlim(0, nx); ax.set_ylim(0, ny); ax.set_zlim(0, nz)
    ax.set_box_aspect((nx, ny, nz))
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_title("Slice position in cube", fontsize=9)
    ax.view_init(elev=22, azim=-58)
    return fig


# --- visualizer widget -----------------------------------------------------
VSTATE = {"data": None, "off": (0, 0, 0), "var": "", "vlim": (0, 1), "busy": False}

vw_file    = widgets.Dropdown(description="File:", layout=W(width="620px"),
                              style={"description_width": "40px"})
vw_refresh = widgets.Button(description="Refresh list", icon="refresh", layout=W(width="140px"))
vw_load    = widgets.Button(description="Load", button_style="primary", layout=W(width="100px"))
vw_orient  = widgets.ToggleButtons(options=[("xy slice", "xy"), ("yz slice", "yz"),
                                            ("xz slice", "xz")], value="xy",
                                   description="Plane:", style={"description_width": "45px"})
vw_pos     = widgets.IntSlider(value=0, min=0, max=1, description="z",
                               continuous_update=False, layout=W(width="520px"),
                               style={"description_width": "30px"})
vw_controls = widgets.VBox([vw_orient, vw_pos])
vw_status = widgets.Output()
vw_img    = widgets.Output()
vw_wire   = widgets.Output()


def _list_saved():
    return sorted(glob.glob(os.path.join(SAVE_DIR, "*.npy")))


def on_refresh(_=None):
    files = _list_saved()
    vw_file.options = [(os.path.basename(f), f) for f in files]
    with vw_status:
        vw_status.clear_output(wait=True)
        print(f"{len(files)} saved cube(s) in {SAVE_DIR}" if files
              else f"No .npy files in {SAVE_DIR} yet - run a fetch in Step 2.")


def _render(*_):
    if VSTATE["busy"] or VSTATE["data"] is None:
        return
    with vw_img:
        vw_img.clear_output(wait=True)
        plot_single_slice(VSTATE["data"], *VSTATE["off"], VSTATE["var"],
                          vw_orient.value, vw_pos.value, VSTATE["vlim"])
    with vw_wire:
        vw_wire.clear_output(wait=True)
        fig = draw_slice_wireframe(VSTATE["data"].shape, vw_orient.value, vw_pos.value)
        display(fig); plt.close(fig)


def _set_pos_axis(orient, default="mid"):
    letter, axis = PERP_AXIS[orient]
    n = VSTATE["data"].shape[axis]
    vw_pos.description = letter
    vw_pos.max = max(n - 1, 0)
    vw_pos.value = n // 2 if default == "mid" else min(default, vw_pos.max)


def on_orient_change(change):
    if VSTATE["busy"] or VSTATE["data"] is None:
        return
    VSTATE["busy"] = True; _set_pos_axis(change["new"]); VSTATE["busy"] = False
    _render()


def on_load(_):
    if not vw_file.value:
        with vw_status:
            vw_status.clear_output(wait=True); print("Pick a file first (Refresh if empty).")
        return
    data = np.load(vw_file.value)
    var, x0, y0, z0 = parse_offsets(vw_file.value)
    VSTATE.update(data=data, off=(x0, y0, z0), var=var,
                  vlim=tuple(np.percentile(data, [2, 98])), busy=True)
    with vw_status:
        vw_status.clear_output(wait=True)
        print(f"Loaded {os.path.basename(vw_file.value)}   shape={data.shape}")
    if 1 in data.shape:                                  # a single plane
        orient = {0: "yz", 1: "xz", 2: "xy"}[list(data.shape).index(1)]
        vw_orient.value = orient; _set_pos_axis(orient, default=0)
        vw_controls.layout.display = "none"
    else:                                                # full 3D cube
        vw_orient.value = "xy"; _set_pos_axis("xy")
        vw_controls.layout.display = ""
    VSTATE["busy"] = False
    _render()


vw_refresh.on_click(on_refresh)
vw_load.on_click(on_load)
vw_orient.observe(on_orient_change, names="value")
vw_pos.observe(_render, names="value")
on_refresh()

viz = widgets.VBox([
    widgets.HTML("<b>Visualize a saved cube</b>"),
    widgets.HBox([vw_file, vw_refresh, vw_load]),
    vw_status, vw_controls,
    widgets.HBox([vw_wire, vw_img]),
], layout=W(padding="12px", max_width="1100px"))
display(viz)